# **Notebook 1**

#### Ian García
#### A01706892

In [ ]:
# Importación de librerias

import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import altair as alt

In [ ]:
# Configuración del estilo visual.
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 8)

# Deshabilitar el limite de 5000 filas de Altair.
alt.data_transformers.enable('default', max_rows=None)

In [ ]:
# Carga y limpieza de datos
url = "https://data.insideairbnb.com/mexico/df/mexico-city/2026-06-15/data/listings.csv.gz"
print("Cargando dataet...")
try:
  df = pl.read_csv(url)
  print("Dataset cargado correctamente")
except Exception as e:
  print(f"Error al cargar el dataset: {e}")

if not df.is_empty():
  df = df.with_columns(
      pl.col("price").str.replace_all(r"\$", "").str.replace_all(",", "").cast(pl.Float64).alias("price")
  ).drop_nulls(subset=["price"]).filter(pl.col("price") > 0)
  print("Limpieza de datos realizada correctamente")
  display(df.head())
else:
  print("El dataset esta vacio")

In [ ]:
# Análisis básico
print("1. Cuál es el pecio promedio por noche?")
print(f"El precio promedio por noche es: {df['price'].mean()}")
print("2. Cuál es el precio máximo por noche?")
print(f"El precio máximo por noche es: {df['price'].max()}")
print("3. Cuál es el precio mínimo por noche?")
print(f"El precio mínimo por noche es: {df['price'].min()}")
print(f"4. Cuales son los tipos de alojamiento?")
print(df.group_by("room_type").agg(pl.len().alias("count")).sort("count", descending=True))
print(f"5. Cual es la alcaldia con mas número de alojamientos?")
print(df.group_by("neighbourhood_cleansed").agg(pl.len().alias("count")).sort("count", descending=True).head(10))
print("6. Quienes son los anfitriones con mas alojamientos?")
print(df.group_by("host_name").agg(pl.len().alias("count")).sort("count", descending=True).head(10))

In [ ]:
# Visualizaciones
price_to_plot_df = df.filter(pl.col("price") < df.select(pl.col("price").quantile(0.95)).item()).to_pandas()

chart_hist = alt.Chart(price_to_plot_df).mark_bar().encode(
    alt.X("price:Q", bin=alt.Bin(maxbins=50), title="Precio por noche"),
    alt.Y("count()", title="Frecuencia"),
    tooltip=[alt.Tooltip('count()', title="Frecuencia"), alt.Tooltip('price:Q', bin=True, title="Rango de Precio")]
).properties(
    title="Distribución de precios",
    width=800,
    height=400
)

chart_hist.show()